# 08. Find Information, Then Answer (RAG) — Embeddings · Similarity · Prompts

> An LLM makes up believable answers even when it doesn't know (hallucination). So we **① find the pieces of text closest to the question, and ② make it answer by looking at those pieces**.
> This is RAG (Retrieval-Augmented Generation: generation boosted by retrieval). Q&A bots for company rules · manuals are built this way.

### What you will learn
1. **Tokenization** — how text becomes numbers
2. **Embeddings** — the meaning of a text as 384 numbers (multilingual-e5-small, INT8)
3. **Cosine similarity** — how close two meanings are
4. Splitting text (chunking), using the match score to say "I don't know", building a prompt

### The whole flow
```
[prepare]   document ─split─▶ chunks ─embed─▶ chunk vectors (384 numbers × number of chunks)
[question]  question ─embed─▶ question vector ─cosine similarity─▶ 2 closest chunks
            (low score → "not in the document")  ─▶ prompt from chunks + question ─▶ LLM ─▶ answer + sources
```

## 1. The embedding model — the tokenizer is an OpenVINO model too
When you import `openvino_tokenizers`, the "text → token numbers" step can also run as an OpenVINO model. That is why there are two files:
`openvino_tokenizer.xml` (text → numbers) and `openvino_model.xml` (numbers → vectors).

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "../..")).resolve()       # program folder (lab.bat tells us)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # model folder
core = ov.Core()
print("Model folder:", MODELS)
print("Devices:", core.available_devices)

import openvino_tokenizers                       # registers the tokenizer operations — just importing it is enough
tcore = ov.Core()                                # the extension attaches to a Core created after the import

E = MODELS / "embed" / "e5-small-int8"
tok = tcore.compile_model(E / "openvino_tokenizer.xml", "CPU")
emb = tcore.compile_model(E / "openvino_model.xml", "CPU")
print("Tokenizer outputs:", [o.any_name for o in tok.outputs])
print("Embedding model inputs:", [i.any_name for i in emb.inputs])

## 2. Tokenization — turning text into piece numbers
The model doesn't know letters. It cuts text into **tokens** (common pieces of text) and turns them into numbers. In `attention_mask`, real tokens are 1 and the padding added to make lengths equal is 0.

In [ ]:
t = tok(["passage: The makerspace opens at 9 a.m. on weekdays", "query: What time does it open?"])
print("input_ids shape:", t["input_ids"].shape)
print("First sentence ids:", t["input_ids"][0])
print("First sentence mask:", t["attention_mask"][0])

## 3. Embeddings — meaning as 384 numbers
1. Each token gives 384 numbers → `[number of sentences, number of tokens, 384]`
2. **Mean pooling**: average only the real tokens (mask=1), so one sentence = 384 numbers
3. **Make the length 1** (normalization): then the dot product of two vectors is exactly the **cosine similarity**

$$\cos(a, b) = \frac{a \cdot b}{\lVert a \rVert\,\lVert b \rVert} \quad (\text{if the lengths are 1, it is } a \cdot b)$$

The closer to 1, the more similar the meaning. The e5 model works best when you put `passage: ` in front of documents and `query: ` in front of questions.

In [ ]:
def embed(texts, kind="passage"):
    """List of texts → [count, 384] (length 1)."""
    t = tok([f"{kind}: {s}" for s in texts])
    feed = {"input_ids": t["input_ids"], "attention_mask": t["attention_mask"]}
    if "token_type_ids" in {i.any_name for i in emb.inputs}:
        feed["token_type_ids"] = np.zeros_like(t["input_ids"])
    out = emb(feed)[emb.output(0)]                    # [count, number of tokens, 384]
    m = t["attention_mask"][..., None].astype(np.float32)
    v = (out * m).sum(1) / m.sum(1)                    # average only the real tokens (mean pooling)
    return v / np.linalg.norm(v, axis=1, keepdims=True)

S = ["A cat sleeps on the sofa", "A kitten takes a nap", "A dog runs in the park", "The stock market rose today"]
v = embed(S)
print("Vector shape:", v.shape, "| length:", np.round(np.linalg.norm(v, axis=1), 3))
print("Similarity table:")
for i, s in enumerate(S):
    print(f"  {s:18s}", " ".join(f"{x:5.2f}" for x in v @ v.T[:, i]))

## 4. The document — splitting it into chunks
If you embed a long text all at once, the meaning gets blurred. Split it **by paragraph**, and split long paragraphs again.
- Too small: one chunk can't hold the whole answer
- Too big: unrelated sentences get mixed in and the score gets fuzzy

In [ ]:
DOC = """
Makerspace Guide

The makerspace is open on weekdays from 9 a.m. to 6 p.m. On Saturdays it closes at 1 p.m. It is closed on Sundays and public holidays.

You can book a 3D printer for up to 3 hours a day. Book on the tablet at the makerspace entrance. If you are more than 10 minutes late, your booking is canceled.

Only people who took the safety training can use the laser cutter. Safety training is every Tuesday at 4 p.m. and lasts 30 minutes.

Soldering is allowed only at workbenches with the ventilation turned on. After using a soldering iron, always put it back in its stand and turn off the power.

Basic filament (PLA) and 3mm plywood are free to use. For other materials, bring your own.
"""

def split(text, size=200):
    """Split on blank lines (paragraphs). Cut long paragraphs at a space within size characters."""
    out = []
    for para in [p.strip() for p in text.split("\n\n") if p.strip()]:
        while len(para) > size:
            cut = para.rfind(" ", 0, size)
            cut = cut if cut > 0 else size
            out.append(para[:cut]); para = para[cut:].strip()
        out.append(para)
    return out

chunks = split(DOC)
V = embed(chunks, "passage")                         # document chunk vectors — make them once and keep them
for i, c in enumerate(chunks):
    print(i, c[:40], "...")
print("Chunk vectors:", V.shape)

## 5. Finding the chunks closest to the question
The dot product of the question vector with every chunk vector = similarity. Take the top `k`.
For a **question that is not in the document**, like "parking lot", even the highest score comes out low — we use this to make it answer "I don't know".

In [ ]:
def search(q, k=2):
    s = V @ embed([q], "query")[0]                    # length is 1, so dot product = cosine similarity
    order = np.argsort(-s)[:k]
    return [(float(s[i]), chunks[i]) for i in order]

for q in ["Is it open on Saturdays?", "What do I need to do to use the laser cutter?", "Is there a parking lot?"]:
    print("Question:", q)
    for score, c in search(q):
        print(f"   {score:.3f}  {c[:50]}")

## 6. Answering with the chunks we found — LLM
We use the VLM (Gemma3 4B) with text only. It is **3.5GB**, so loading it the first time takes a while (if edge-lab is running, close it first).

The **prompt** makes "answer only from the document" very clear, and puts things in this order: document → question → "Answer:".
`do_sample=False` makes the same question always get the same answer.

In [ ]:
import openvino_genai as og

VLM_DIR = MODELS / "vlm" / "gemma3-4b-int4"
DEVICE = "GPU" if any(d.startswith("GPU") for d in core.available_devices) else "CPU"
t0 = time.perf_counter()
llm = og.VLMPipeline(str(VLM_DIR), DEVICE)
print(f"LLM loaded: {time.perf_counter() - t0:.0f} s ({DEVICE})")

cfg = og.GenerationConfig()
cfg.max_new_tokens = 160
cfg.do_sample = False
cfg.repetition_penalty = 1.15

In [ ]:
MIN_SCORE = 0.60        # below this, we treat the question as not in the document (tuned for e5-small INT8 on Korean text — print the scores and adjust for English)

def make_prompt(q, hits):
    context = "\n".join(c for _, c in hits)
    return (f"Answer the question briefly in English, using only the document below.\n--- Document ---\n{context}\n--- End ---\n"
            f"Question: {q}\nAnswer:")

def answer(q):
    hits = search(q, 2)
    if hits[0][0] < MIN_SCORE:
        return "I couldn't find that in the document.", hits
    return str(llm.generate(make_prompt(q, hits), generation_config=cfg)).strip(), hits

print("— The text that actually goes into the LLM —")
print(make_prompt("When is the safety training?", search("When is the safety training?", 2)))

In [ ]:
for q in ["How many hours can I use the 3D printer?", "When is the safety training?", "Is there a parking lot?"]:
    t0 = time.perf_counter()
    a, hits = answer(q)
    print(f"Q: {q}\nA: {a}\n   (match score {hits[0][0]:.2f} · {time.perf_counter() - t0:.1f} s)\n")

## Common errors
| Symptom | Cause · fix |
|---|---|
| `Cannot create ... SpecialTokensSplit` | You forgot `import openvino_tokenizers`, or you used a Core created before it |
| All scores are similarly low | You didn't add `passage: ` / `query: ` |
| "Couldn't find it" for a question that is in the document | `MIN_SCORE` is too high → print the scores and adjust |
| The answer doesn't match the document | The prompt is missing "only from the document", or the chunks are too big |
| Out of memory | edge-lab has the same model loaded → close it and try again |

## Summary
- Embedding = a vector that holds meaning. If you make the length 1, dot product = cosine similarity.
- RAG = finding (embeddings · similarity) + answering (LLM). The **search score** tells you when to say "I don't know".
- Chunk size · score threshold · prompt decide the quality.

---
# 🛠 Mini project — Q&A bot for your own document

**What to build**: a bot that reads one text file and, when you type a question, shows the answer together with its **source chunks**. Type an empty line to stop.

**What you use**: `split` · `embed` · `search` · `answer` (the functions built in this notebook, as they are)

**Steps**
1. Read the document — `my_doc.txt` if it exists, otherwise the `DOC` above
2. Make the chunks · vectors
3. Repeat: question → answer + sources + scores

In [ ]:
doc_path = Path("my_doc.txt")                     # put your own document next to this notebook and it will be used
text = doc_path.read_text(encoding="utf-8") if doc_path.exists() else DOC
chunks = split(text)
V = embed(chunks, "passage")
print(f"Document: {doc_path if doc_path.exists() else 'example DOC'} · {len(chunks)} chunks")

while True:
    q = input("Question (just press Enter to quit): ").strip()
    if not q:
        break
    a, hits = answer(q)
    print(f"\nAnswer: {a}")
    for score, c in hits:
        print(f"  source {score:.2f} | {c[:70]}")
    print()

**Try more**
- Save each answer and its sources to `qa_log.txt`, adding to the file for every question
- Show the source chunks with the question's words marked in `[ ]`
- Read several documents (every `.txt` in a folder), and show the file name with each source